In [ ]:
import GtoTmodel as GtoTmodel
import torch
import torch.nn as nn
import torch.optim as optim

In [12]:
torch.manual_seed(1337)
torch.cuda.manual_seed(1337)
embed_dim = 16  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers = 2  # Number of transformer layers
dropout = 0.1  # Dropout rate

#graph_colomns=5
num_components=5
batch_size = 16  # Batch size

graph_input_dim = 10  # Number of colomns in the graph
text_vocab_size = 26  # Vocabulary size for text


Importing the model

In [13]:
model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim, 
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

c:\Users\MSI\miniconda3\envs\ml\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [15]:
learning_rate = 0.001
num_epochs = 10000

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Sample trainig set

In [16]:

num_ciruits = 320
graph_dataset =  torch.randn(num_ciruits,graph_input_dim, graph_input_dim )
text_dataset = torch.randint(0, text_vocab_size, (num_ciruits,graph_input_dim))

graph_dataset = graph_dataset.to(device)
text_dataset = text_dataset.to(device)


In [17]:
graph_dataset.shape , text_dataset.shape

(torch.Size([320, 10, 10]), torch.Size([320, 10]))

Split data in to test and train set

In [ ]:
from sklearn.model_selection import train_test_split

graph_train, graph_test, text_train, text_test = train_test_split(graph_dataset, text_dataset, test_size=0.2)



In [ ]:
# Training loop
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    total_loss = 0  # Track total loss for the epoch

    for i in range(0, len(graph_train), batch_size):
        # Get the current batch of data
        graph_data = graph_train[i:i+batch_size]
        text = text_train[i:i+batch_size]

        # Prepare input and target
        text_input = text[:, :-1]  # All but the last token
        target = text[:, -1]  # Last token as the target

        # Forward pass
        output = model(graph_data, text_input)
        output = output[:, -1, :]  # Select the last timestep's predictions

        # Compute loss
        loss = criterion(output, target)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Accumulate loss
        total_loss += loss.item()

    # Print loss for the epoch
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {total_loss / len(graph_train):.4f}")

RuntimeError: Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got torch.cuda.FloatTensor instead (while checking arguments for embedding)

In [ ]:
# graph_data = graph_train[i:i+batch_size]
# text = text_train[i:i+batch_size]
# text_input = text[:, :-1]
# print(text_input)
# target = text[:,-2:-1].reshape(-1)
# print(target)
# output = model(graph_data[:, :-1], text_input)  # Exclude the last token for input
# output = output.reshape(-1, text_vocab_size)  # Reshape for loss calculation
# print(output.shape)
# print(output)

# Training loop 
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    
    # Forward pass
    for i in range(0, len(graph_train), batch_size):
        graph_data = graph_train[i:i+batch_size]
        text = text_train[i:i+batch_size]
        print("Graph data shape",graph_data.shape)
        print("text data shape",text[0].shape)
        
        # Prepare input and target
        text_input = text[:, :-1]  # All but the last token
        target = text[:, -1]  # Last token as the target
        print("Text input shape",text_input.shape)
        print("Target shape",target.shape)
        # Forward pass
        output = model(graph_data, text_input)
        
        # Select the last timestep's predictions
        output = output[:, -1, :]  # Shape: [batch_size, vocab_size]
        
        # Compute loss
        loss = criterion(output, target)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Print loss for the epoch
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")



Graph data shape torch.Size([16, 10, 10])
text data shape torch.Size([10])
Text input shape torch.Size([16, 9])
Target shape torch.Size([16])
Graph data shape torch.Size([16, 10, 10])
text data shape torch.Size([10])
Text input shape torch.Size([16, 9])
Target shape torch.Size([16])
Graph data shape torch.Size([16, 10, 10])
text data shape torch.Size([10])
Text input shape torch.Size([16, 9])
Target shape torch.Size([16])
Graph data shape torch.Size([16, 10, 10])
text data shape torch.Size([10])
Text input shape torch.Size([16, 9])
Target shape torch.Size([16])
Graph data shape torch.Size([16, 10, 10])
text data shape torch.Size([10])
Text input shape torch.Size([16, 9])
Target shape torch.Size([16])
Graph data shape torch.Size([16, 10, 10])
text data shape torch.Size([10])
Text input shape torch.Size([16, 9])
Target shape torch.Size([16])
Graph data shape torch.Size([16, 10, 10])
text data shape torch.Size([10])
Text input shape torch.Size([16, 9])
Target shape torch.Size([16])
Graph 

KeyboardInterrupt: 

In [ ]:
# Validation phase
model.eval()  # Switch to evaluation mode
# Initialize variables to track metrics
correct = 0
total = 0
all_labels = []
all_predictions = []
cumulative_confidence = 0  # Track prediction confidences

with torch.no_grad():
    for i in range(0, len(graph_test), batch_size):
        # Move inputs and labels to the device (CPU or GPU)
        graph_data = graph_test[i:i+batch_size]
        text = text_test[i:i+batch_size]
        text_input = text[:, :-1]
        labels = text[:, -2:-1].reshape(-1)
        graph_data, text_input, labels = graph_data.to(device), text_input.to(device), labels.to(device)
        
        outputs = model(graph_data[:, :-1], text_input)  # Exclude the last token for input
        outputs = outputs.reshape(-1, text_vocab_size)
        
        # Get the predicted class index (0 or 1)
        probabilities = torch.softmax(outputs, dim=1)
        max_probabilities, predicted = torch.max(probabilities, 1)
        
        # Track prediction confidence
        cumulative_confidence += max_probabilities.mean().item()
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()  # Compare with true labels
        
        # Store all labels and predictions for further metrics calculation
        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

# Calculate accuracy as the percentage of correct predictions
accuracy = 100 * correct / total

# Calculate average prediction confidence
avg_confidence = 100 * cumulative_confidence / (total // batch_size)

# Calculate the confusion matrix
conf_matrix = confusion_matrix(all_labels, all_predictions)
tn, fp, fn, tp = conf_matrix.ravel()

# Print the metrics
print(f'Validation Accuracy: {accuracy:.2f}%')
print(f'Average Prediction Confidence: {avg_confidence:.2f}%')
print(f'True Negatives (Real identified as Real): {tn}')
print(f'False Positives (Real identified as Fake): {fp}')
print(f'False Negatives (Fake identified as Real): {fn}')
print(f'True Positives (Fake identified as Fake): {tp}')
print('Confusion Matrix:')
print(conf_matrix)

# Optional: Calculate additional metrics
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f'\nPrecision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1_score:.4f}')

Input graph_data shape: torch.Size([2, 4, 10])
graph_encoded shape: torch.Size([2, 4, 16])
graph_encoded permuted shape: torch.Size([4, 2, 16])


RuntimeError: The size of tensor a (8) must match the size of tensor b (2) at non-singleton dimension 0

In [ ]:
import torch.backends.cudnn as cudnn
cudnn.benchmark = True
import torch.backends.cudnn.deterministic = True
import numpy as np
import pandas as pd
import random
import os
import argparse
import time
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import networkx as nx
import json
import pickle
import copy
import torch_geometric
import torch_geometric.transforms as T
import torch_geometric.nn as pyg_nn
import torch_geometric.data as pyg_data
import torch_geometric.utils as pyg_utils
import torch_geometric.datasets as pyg_datasets
import torch_geometric.loader as pyg_loader
import torch_geometric.nn as pyg_nn
import torch_geometric.utils as pyg_utils